# Router Chain: 实现条件判断的大模型调用


SequentialChain = 按顺序一步步执行（串行流水线）
RouterChain = 按问题类型「分类分流」，走不同分支链

# 旧版 RouterChain 用法
# 定义子 Prompt → 组装路由链 → invoke 自动路由

# 伪代码结构
from langchain.chains.router import MultiPromptChain

# 1. 定义各个场景的提示词
prompt_infos = [
    {"name":"tech", "description":"技术问题", "prompt":技术模板},
    {"name":"life", "description":"生活闲聊", "prompt":生活模板}
]

# 2. 构建路由链
router_chain = MultiPromptChain(
    llm=llm,
    prompt_infos=prompt_infos,
    verbose=True
)

特点：
同样继承 Chain 基类
同样支持 verbose=True
不用自己写判断，内部自动生成路由提示词做分类

# 新版语法
新版不再用 MultiPromptChain 类，用这套组合替代：

定义各个分支链
写一个路由分类 Prompt → 输出类别
用 Runnable 条件分支 匹配对应链
思想和旧 RouterChain 完全一样，只是换成管道写法。

In [4]:
# ======================  导入依赖 ======================
from langchain_core.prompts import PromptTemplate  # 提示词模板
from langchain_community.chat_models import ChatZhipuAI        # 智谱模型
from langchain_core.output_parsers import StrOutputParser  # 输出解析器
from langchain_core.runnables import RunnableBranch, RunnablePassthrough  
import os
from dotenv import load_dotenv
load_dotenv()


# ======================  初始化 智谱模型 ======================
llm = ChatZhipuAI(model="glm-4.5-air", api_key=os.getenv("ZHIPUAI_API_KEY"), temperature=0.3)
parser = StrOutputParser()

# 1. 定义两个业务分支链
# 技术分支
tech_prompt = PromptTemplate.from_template("你是技术专家，解答技术问题：{input}")
tech_chain = tech_prompt | llm | parser

# 生活分支
life_prompt = PromptTemplate.from_template("你是生活助手，闲聊解答生活问题：{input}")
life_chain = life_prompt | llm | parser

# 2. 路由分类 Prompt：判断输入属于哪一类
route_prompt = PromptTemplate.from_template("""
请判断用户问题属于【技术】还是【生活】，只返回两个字：技术 / 生活
用户问题：{input}
""")

# 分类链：输出 技术 / 生活
classify_chain = route_prompt | llm | parser

# 3. 条件分支路由（新版 RouterChain 核心）
router_chain = RunnableBranch(
    (lambda x: "技术" in x["type"], tech_chain),
    (lambda x: "生活" in x["type"], life_chain),
    # 默认分支
    life_chain
)

# 4. 组装整体链
full_chain = (
     RunnablePassthrough.assign(input=lambda x: x["question"])
    .assign(type=classify_chain) # ← 这一步给字典加上了 type 字段！
    | router_chain
)

# # 1. 输入适配
# step1 = RunnablePassthrough.assign(input=lambda x: x["question"])

# # 2. 生成分类，同时保留input
# step2 = RunnablePassthrough.assign(
#     category = route_prompt | llm | parser
# )

# # 3. 分支路由
# step3 = router_chain

# # 拼装整体链
# full_chain = step1 | step2 | step3



# 调用
res = full_chain.invoke({"question": "Python 装饰器怎么用？"}, config={"verbose":True})
print(res)


好的，作为技术专家，我来详细解释一下 Python 装饰器的概念、原理和使用方法。装饰器是 Python 中非常强大且优雅的特性，它允许你**在不修改原函数代码的情况下，动态地扩展或修改函数的行为**。

## 核心概念

1.  **函数是一等公民：** 在 Python 中，函数就像其他对象（如整数、字符串、列表）一样，可以被：
    *   赋值给变量
    *   作为参数传递给其他函数
    *   作为其他函数的返回值
    *   存储在数据结构中
2.  **闭包：** 装饰器通常利用闭包来实现。闭包是指一个函数（内部函数）能够“记住”并访问其定义时所在的词法作用域（即使该作用域已经结束）。
3.  **装饰器本质：** 装饰器本质上是一个**可调用对象**（通常是函数，也可以是类），它接受一个函数作为参数，并返回一个新的可调用对象（通常是修改后的函数）。

## 装饰器的基本语法

```python
@decorator_name
def function_to_decorate():
    # 函数体
    pass
```

这行代码**等价于**：

```python
def function_to_decorate():
    # 函数体
    pass

function_to_decorate = decorator_name(function_to_decorate)
```

## 工作原理

1.  当 Python 解释器遇到 `@decorator_name` 时，它会立即执行 `decorator_name` 这个函数（或可调用对象）。
2.  `decorator_name` 函数接收 `function_to_decorate`（即被装饰的函数）作为参数。
3.  `decorator_name` 内部通常会定义一个新的函数（内部函数），这个新函数会：
    *   执行一些额外的操作（如日志记录、权限检查、计时等）。
    *   调用原始的 `function_to_decorate` 函数（可能传入参数，也可能不传）。
    *   可能对原始函数的返回值进行处理。
    *   返回这个新的内部函数。
4.  最后，原始函数名 `function_to_decorate` 被重新赋值为

In [5]:
# 调用
res = full_chain.invoke({"question": "现在好饿啊"}, config={"verbose":True})
print(res)

哈哈，饥饿感来袭！别慌，我来帮你“解救”你的胃～ 🍔🍜🍎

### **快速救急方案（5分钟内搞定）：**
1. **零食补给站**  
   - 家里有坚果/酸奶/香蕉？来点高蛋白+碳水组合，顶饿又健康！  
   - 没存货？冲杯热牛奶/燕麦片，暖胃又快速～  
2. **外卖闪电战**  
   - 打开APP搜“最快送达”，优先选：  
     ✅ 粥/汤粉（清淡不撑）  
     ✅ 汉堡/饭团（方便快捷）  
     ✅ 水果捞（解馋又补水）  
3. **厨房魔法（有食材的话）**  
   - 鸡蛋+剩米饭 → 炒个蛋炒饭（5分钟香喷喷！）  
   - 面条+酱油+葱花 → 葱油面灵魂一拌！  

### **如果还能等半小时（顺便补充能量知识）：**
- **为什么突然饿？**  
  可能是：①饭点到了 ②血糖波动 ③太忙忘了吃… 下次随身带小零食预防！  
- **健康小贴士**：  
  饿时别狂吃高糖食物（如蛋糕），会血糖过山车！选复合碳水（全麦面包/红薯）更持久～  

### **终极提问：**  
🍜 **你现在最想吃什么？** 是热汤面、炸鸡、还是水果沙拉？告诉我，我帮你找做法/外卖攻略！ 😋  

（悄悄说：我冰箱里永远囤着黑巧克力，饿到发慌时来两块，瞬间回血！）
